[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/ia_p26/blob/main/clase/17_multi_armed_bandits/notebooks/03_bayesianos_y_comparacion.ipynb)

In [ ]:
# !pip install numpy matplotlib scipy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import brentq

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

COLORS = {
    "red": "#E94F37", "orange": "#F28C28", "blue": "#2E86AB",
    "green": "#27AE60", "purple": "#8E44AD", "gray": "#7F8C8D",
}
ARM_COLORS = [COLORS["red"], COLORS["orange"], COLORS["blue"]]
ARM_NAMES = ["A", "B", "C"]

# Notebook 3: Algoritmos Bayesianos y Comparacion Final

En este notebook implementamos **Thompson Sampling** (el enfoque bayesiano para
bandidos) y **EXP3** (para el caso adversarial), y realizamos una **comparacion
exhaustiva** de los cinco algoritmos del modulo:

1. $\varepsilon$-Greedy
2. UCB1
3. KL-UCB
4. Thompson Sampling (Bernoulli y Gaussiano)
5. EXP3

Antes de implementar Thompson, necesitamos entender su ingrediente central:
la **distribucion Beta** y la conjugacion Beta-Bernoulli.

---
## Seccion 1: Entorno de bandidos (autocontenido)

In [ ]:
class BanditEnvironment:
    """Multi-armed bandit environment with Bernoulli or Gaussian arms."""

    def __init__(self, means, reward_type="bernoulli", sigma=1.5):
        self.means = np.array(means, dtype=float)
        self.K = len(means)
        self.reward_type = reward_type
        self.sigma = sigma  # only used for Gaussian
        self.best_mean = np.max(self.means)

    def pull(self, arm, rng=None):
        """Pull an arm and return a stochastic reward."""
        if rng is None:
            rng = np.random.default_rng()
        if self.reward_type == "bernoulli":
            return float(rng.random() < self.means[arm])
        else:  # gaussian
            return rng.normal(self.means[arm], self.sigma)


# Canonical problems from Section 17.1
CANONICAL_A = BanditEnvironment([0.3, 0.5, 0.7], reward_type="bernoulli")
CANONICAL_B = BanditEnvironment([1.0, 2.0, 3.0], reward_type="gaussian", sigma=1.5)

print("Canonical A (Bernoulli):", CANONICAL_A.means, "| best =", CANONICAL_A.best_mean)
print("Canonical B (Gaussian): ", CANONICAL_B.means, "| best =", CANONICAL_B.best_mean)

---
## Seccion 2: Explorando la distribucion Beta

La distribucion $\text{Beta}(\alpha, \beta)$ describe la incertidumbre sobre
una probabilidad $\theta \in [0, 1]$. Sus parametros tienen una interpretacion
como **conteos**:

- $\alpha \approx$ exitos observados + 1
- $\beta \approx$ fracasos observados + 1

Veamos como cambia su forma con distintos parametros.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=True)

beta_params = [
    (1, 1, 'Beta(1,1) — Uniforme\n"No se nada"'),
    (2, 5, 'Beta(2,5) — Sesgada izquierda\n"Probablemente bajo"'),
    (5, 2, 'Beta(5,2) — Sesgada derecha\n"Probablemente alto"'),
    (10, 10, 'Beta(10,10) — Campana centrada\n"Bastante seguro de ~0.5"'),
    (30, 70, 'Beta(30,70) — Pico estrecho\n"Muy seguro de ~0.3"'),
    (1, 3, 'Beta(1,3) — Sesgada izquierda\n"Probablemente bajo (~0.25)"'),
]

x = np.linspace(0, 1, 500)

for ax, (a, b, title) in zip(axes.flat, beta_params):
    pdf = stats.beta.pdf(x, a, b)
    ax.fill_between(x, pdf, alpha=0.3, color=COLORS["blue"])
    ax.plot(x, pdf, color=COLORS["blue"], lw=2)
    mean_val = a / (a + b)
    ax.axvline(mean_val, color=COLORS["red"], ls="--", lw=1.5,
               label=f"media = {mean_val:.2f}")
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=9)
    ax.set_xlim(0, 1)

fig.suptitle("Distribucion Beta: como cambia la forma con los parametros",
             fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

### Ejercicio: tus propios parametros

Cambia los valores de `alpha` y `beta` para responder:

1. Como se ve una Beta que dice "estoy 90% seguro de que $\theta \approx 0.9$"?
2. Que pasa cuando $\alpha = \beta$ y ambos son muy grandes?

In [ ]:
alpha = 2   # <-- CHANGE THIS
beta_ = 2   # <-- CHANGE THIS

x = np.linspace(0, 1, 500)
pdf = stats.beta.pdf(x, alpha, beta_)

plt.figure(figsize=(8, 4))
plt.fill_between(x, pdf, alpha=0.3, color=COLORS["purple"])
plt.plot(x, pdf, color=COLORS["purple"], lw=2)
plt.axvline(alpha / (alpha + beta_), color=COLORS["red"], ls="--",
            label=f"media = {alpha / (alpha + beta_):.3f}")
plt.title(f"Beta({alpha}, {beta_})", fontsize=13)
plt.xlabel(r"$\theta$")
plt.ylabel("Densidad")
plt.legend()
plt.tight_layout()
plt.show()

---
## Seccion 3: Conjugacion Beta-Bernoulli

La magia de la distribucion Beta para bandidos Bernoulli: si nuestro prior sobre
$\theta$ es $\text{Beta}(\alpha, \beta)$ y observamos $r \in \{0, 1\}$:

$$\text{Posterior} = \begin{cases} \text{Beta}(\alpha + 1, \beta) & \text{si } r = 1 \\ \text{Beta}(\alpha, \beta + 1) & \text{si } r = 0 \end{cases}$$

El posterior sigue siendo Beta. Solo sumamos 1 al parametro correspondiente.
Veamos como evoluciona el posterior conforme observamos datos.

In [ ]:
# Simulate a sequence of Bernoulli observations with true theta = 0.7
true_theta = 0.7
rng = np.random.default_rng(42)
observations = rng.random(20) < true_theta  # 20 observations

# Track posterior evolution
alpha_post, beta_post = 1, 1  # start with Beta(1,1) = Uniform
snapshots = [(0, alpha_post, beta_post)]  # (n_obs, alpha, beta)

for i, obs in enumerate(observations):
    if obs:
        alpha_post += 1
    else:
        beta_post += 1
    if (i + 1) in [1, 3, 5, 10, 20]:
        snapshots.append((i + 1, alpha_post, beta_post))

# Plot posterior at each snapshot
fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=True)
x = np.linspace(0, 1, 500)
colors_snap = [COLORS["gray"], COLORS["purple"], COLORS["blue"],
               COLORS["green"], COLORS["orange"], COLORS["red"]]

for ax, (n, a, b), c in zip(axes.flat, snapshots, colors_snap):
    pdf = stats.beta.pdf(x, a, b)
    ax.fill_between(x, pdf, alpha=0.3, color=c)
    ax.plot(x, pdf, color=c, lw=2)
    ax.axvline(true_theta, color="black", ls=":", lw=1.5, label=r"$\theta$ real = 0.7")
    ax.axvline(a / (a + b), color=c, ls="--", lw=1.5,
               label=f"media posterior = {a/(a+b):.2f}")
    ax.set_title(f"Despues de {n} obs: Beta({a}, {b})", fontsize=10)
    ax.legend(fontsize=8)
    ax.set_xlim(0, 1)

fig.suptitle("Evolucion del posterior Beta-Bernoulli",
             fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

print(f"Observaciones: {observations.astype(int)}")
print(f"Exitos: {observations.sum()}, Fracasos: {(~observations).sum()}")

**Observa** como el posterior pasa de plano (Uniforme) a concentrarse alrededor
del valor real $\theta = 0.7$. Con mas datos, la varianza del posterior decrece
y nuestra incertidumbre se reduce.

---
## Seccion 4: Thompson Sampling para Bernoulli

El algoritmo:
1. Mantener un posterior $\text{Beta}(\alpha_i, \beta_i)$ por brazo
2. Muestrear $\theta_i \sim \text{Beta}(\alpha_i, \beta_i)$ para cada brazo
3. Jalar el brazo con mayor $\theta_i$
4. Actualizar el posterior del brazo jalado

Los marcadores `[T1]`-`[T5]` corresponden al pseudocodigo de la Seccion 17.4.

In [ ]:
class ThompsonBernoulli:
    """Thompson Sampling for Bernoulli bandits with Beta prior."""

    def __init__(self, K):
        self.K = K
        self.alpha = np.ones(K)  # [T1] prior Beta(1,1) = Uniform
        self.beta = np.ones(K)   # [T1]

    def select(self, rng):
        # [T2] Sample from each posterior
        samples = rng.beta(self.alpha, self.beta)
        # [T3] Select arm with highest sample
        return int(np.argmax(samples))

    def update(self, arm, reward):
        if reward == 1:
            self.alpha[arm] += 1  # [T4] success: increment alpha
        else:
            self.beta[arm] += 1   # [T5] failure: increment beta


print("ThompsonBernoulli listo.")

---
## Seccion 5: Traza manual — Thompson en Canonical A

Reproducimos los primeros 8 pasos de la traza de la Seccion 17.4 (seed=42).
Comparemos con la tabla del material de clase.

In [ ]:
env = BanditEnvironment([0.3, 0.5, 0.7], reward_type="bernoulli")
ts = ThompsonBernoulli(3)
rng = np.random.default_rng(42)

print(f"{'t':>3} | {'Beta_A':>10} | {'theta_A':>8} | {'Beta_B':>10} | "
      f"{'theta_B':>8} | {'Beta_C':>10} | {'theta_C':>8} | {'Arm':>3} | {'r':>3}")
print("-" * 95)

for t in range(1, 9):
    # Sample from posteriors
    samples = rng.beta(ts.alpha, ts.beta)
    arm = int(np.argmax(samples))

    # Pull arm and get reward (use same rng for reproducibility with section 17.4)
    reward = env.pull(arm, rng)

    beta_strs = [f"({int(ts.alpha[i])},{int(ts.beta[i])})" for i in range(3)]

    print(f"{t:>3} | {beta_strs[0]:>10} | {samples[0]:>8.3f} | "
          f"{beta_strs[1]:>10} | {samples[1]:>8.3f} | "
          f"{beta_strs[2]:>10} | {samples[2]:>8.3f} | "
          f"{ARM_NAMES[arm]:>3} | {int(reward):>3}")

    ts.update(arm, reward)

---
## Seccion 6: Visualizacion de posteriores

Ejecutamos Thompson Sampling por $T = 200$ rondas y visualizamos los posteriores
Beta en cuatro momentos: $t = 1, 10, 50, 200$.

In [ ]:
# Run Thompson for T=200
env = BanditEnvironment([0.3, 0.5, 0.7], reward_type="bernoulli")
ts = ThompsonBernoulli(3)
rng = np.random.default_rng(42)

# Save posterior snapshots
snapshot_times = [1, 10, 50, 200]
posterior_snapshots = {}

for t in range(1, 201):
    samples = rng.beta(ts.alpha, ts.beta)
    arm = int(np.argmax(samples))
    reward = env.pull(arm, rng)
    ts.update(arm, reward)

    if t in snapshot_times:
        posterior_snapshots[t] = (ts.alpha.copy(), ts.beta.copy())

# Plot
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=False)
x = np.linspace(0, 1, 500)

for ax, t in zip(axes, snapshot_times):
    alphas, betas = posterior_snapshots[t]
    for i in range(3):
        pdf = stats.beta.pdf(x, alphas[i], betas[i])
        ax.plot(x, pdf, color=ARM_COLORS[i], lw=2,
                label=f"{ARM_NAMES[i]}: Beta({int(alphas[i])},{int(betas[i])})")
        ax.fill_between(x, pdf, alpha=0.15, color=ARM_COLORS[i])
    # True means
    for i, mu in enumerate(env.means):
        ax.axvline(mu, color=ARM_COLORS[i], ls=":", lw=1, alpha=0.7)
    ax.set_title(f"t = {t}", fontsize=12)
    ax.legend(fontsize=8)
    ax.set_xlim(0, 1)
    ax.set_xlabel(r"$\theta$")

axes[0].set_ylabel("Densidad")
fig.suptitle("Evolucion de posteriores Beta durante Thompson Sampling",
             fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

---
## Seccion 7: Thompson Sampling para Gaussianas

Para recompensas $r \sim \mathcal{N}(\mu_i, \sigma^2)$ con $\sigma$ conocida,
usamos la conjugacion **Normal-Normal**. El posterior sobre $\mu_i$ es:

$$\mu_i \mid \text{datos} \sim \mathcal{N}(\mu_n, \sigma_n^2)$$

con actualizacion por precision (inverso de la varianza).

In [ ]:
class ThompsonGaussian:
    """Thompson Sampling for Gaussian bandits with Normal-Normal conjugacy."""

    def __init__(self, K, mu_prior=0.0, sigma_prior=10.0, sigma_known=1.5):
        self.K = K
        self.sigma_known = sigma_known
        self.tau_data = 1.0 / sigma_known**2  # precision of observations

        # Prior: N(mu_prior, sigma_prior^2) for each arm
        self.mu = np.full(K, mu_prior)
        self.tau = np.full(K, 1.0 / sigma_prior**2)  # precision of posterior

    def select(self, rng):
        # Sample from each posterior N(mu_i, 1/tau_i)
        sigma_post = 1.0 / np.sqrt(self.tau)
        samples = rng.normal(self.mu, sigma_post)
        return int(np.argmax(samples))

    def update(self, arm, reward):
        # Precision-weighted update
        tau_new = self.tau[arm] + self.tau_data
        self.mu[arm] = (self.tau[arm] * self.mu[arm] +
                        self.tau_data * reward) / tau_new
        self.tau[arm] = tau_new


# Quick test on Canonical B
env_b = CANONICAL_B
ts_g = ThompsonGaussian(3, mu_prior=0.0, sigma_prior=10.0, sigma_known=1.5)
rng = np.random.default_rng(42)

for t in range(200):
    arm = ts_g.select(rng)
    reward = env_b.pull(arm, rng)
    ts_g.update(arm, reward)

print("Posterior means after T=200:")
for i in range(3):
    sigma_post = 1.0 / np.sqrt(ts_g.tau[i])
    print(f"  Arm {ARM_NAMES[i]}: mu_post = {ts_g.mu[i]:.3f}, "
          f"sigma_post = {sigma_post:.3f} (true mu = {env_b.means[i]})")

---
## Seccion 8: EXP3 — Bandidos adversariales

EXP3 (Exponential-weight algorithm for Exploration and Exploitation) no asume
recompensas estocasticas. Usa:

1. **Pesos multiplicativos** por brazo
2. **Mezcla con exploracion uniforme** ($\gamma / K$)
3. **Importance weighting** para estimar perdidas

Los marcadores `[E1]`-`[E4]` corresponden al pseudocodigo de la Seccion 17.6.

**Nota**: EXP3 trabaja con **perdidas** (no recompensas). Para compatibilidad
con nuestro entorno de bandidos, convertimos recompensas a perdidas: $\ell = 1 - r$
para Bernoulli, y normalizamos para Gaussiano.

In [ ]:
class EXP3:
    """EXP3 algorithm for adversarial bandits."""

    def __init__(self, K, gamma=0.1):
        self.K = K
        self.gamma = gamma
        self.weights = np.ones(K)  # [E1] equal initial weights
        self.probs = np.ones(K) / K

    def select(self, rng):
        # [E2] Mix: weights + uniform exploration
        W = self.weights.sum()
        self.probs = (1 - self.gamma) * self.weights / W + self.gamma / self.K
        return int(rng.choice(self.K, p=self.probs))

    def update(self, arm, reward):
        # Convert reward to loss in [0, 1]
        loss = 1.0 - np.clip(reward, 0, 1)
        # [E3] Importance-weighted loss estimator
        loss_hat = loss / self.probs[arm]
        # [E4] Multiplicative weight update
        eta = self.gamma / self.K
        self.weights[arm] *= np.exp(-eta * loss_hat)
        # Prevent numerical underflow
        self.weights = np.maximum(self.weights, 1e-300)


print("EXP3 listo.")

---
## Seccion 9: Epsilon-Greedy y UCB1 (redefinicion)

Para la comparacion final, redefinimos los algoritmos de los notebooks anteriores.

In [ ]:
class EpsilonGreedy:
    """Epsilon-greedy with constant epsilon."""

    def __init__(self, K, epsilon=0.1):
        self.K = K
        self.epsilon = epsilon
        self.Q = np.zeros(K)  # estimated mean reward
        self.N = np.zeros(K)  # pull counts

    def select(self, rng):
        if rng.random() < self.epsilon:
            return int(rng.integers(self.K))
        return int(np.argmax(self.Q))

    def update(self, arm, reward):
        self.N[arm] += 1
        self.Q[arm] += (reward - self.Q[arm]) / self.N[arm]


class UCB1:
    """UCB1 algorithm."""

    def __init__(self, K):
        self.K = K
        self.Q = np.zeros(K)
        self.N = np.zeros(K)
        self.t = 0

    def select(self, rng):
        self.t += 1
        # Force exploration of unvisited arms
        for i in range(self.K):
            if self.N[i] == 0:
                return i
        # UCB1 formula
        ucb_values = self.Q + np.sqrt(2 * np.log(self.t) / self.N)
        return int(np.argmax(ucb_values))

    def update(self, arm, reward):
        self.N[arm] += 1
        self.Q[arm] += (reward - self.Q[arm]) / self.N[arm]


class KLUCB:
    """KL-UCB for Bernoulli bandits."""

    def __init__(self, K):
        self.K = K
        self.Q = np.zeros(K)
        self.N = np.zeros(K)
        self.t = 0

    @staticmethod
    def _kl_bernoulli(p, q):
        """KL divergence between Bernoulli(p) and Bernoulli(q)."""
        p = np.clip(p, 1e-10, 1 - 1e-10)
        q = np.clip(q, 1e-10, 1 - 1e-10)
        return p * np.log(p / q) + (1 - p) * np.log((1 - p) / (1 - q))

    def _kl_ucb_value(self, arm):
        """Find max q in [mu_hat, 1] such that KL(mu_hat, q) <= log(t)/N."""
        p = np.clip(self.Q[arm], 1e-10, 1 - 1e-10)
        threshold = np.log(self.t) / self.N[arm]
        try:
            q_max = brentq(lambda q: self._kl_bernoulli(p, q) - threshold,
                           p, 1 - 1e-10)
        except ValueError:
            q_max = 1.0
        return q_max

    def select(self, rng):
        self.t += 1
        for i in range(self.K):
            if self.N[i] == 0:
                return i
        ucb_values = np.array([self._kl_ucb_value(i) for i in range(self.K)])
        return int(np.argmax(ucb_values))

    def update(self, arm, reward):
        self.N[arm] += 1
        self.Q[arm] += (reward - self.Q[arm]) / self.N[arm]


print("EpsilonGreedy, UCB1, KLUCB listos.")

---
## Seccion 10: Gran comparacion — 200 runs, ambos canonicos

Ejecutamos los 5 algoritmos sobre los dos problemas canonicos con 200 runs
independientes. Usamos `seed = 42 + run_index` para reproducibilidad total.

In [ ]:
def run_experiment(env, algo_class, algo_kwargs, T=1000, n_runs=200, seed_base=42):
    """Run n_runs of an algorithm on an environment, return regret trajectories."""
    all_regret = np.zeros((n_runs, T))
    all_arms = np.zeros((n_runs, T), dtype=int)

    for run in range(n_runs):
        rng = np.random.default_rng(seed_base + run)
        algo = algo_class(env.K, **algo_kwargs)
        cum_regret = 0.0

        for t in range(T):
            arm = algo.select(rng)
            reward = env.pull(arm, rng)
            algo.update(arm, reward)

            instant_regret = env.best_mean - env.means[arm]
            cum_regret += instant_regret
            all_regret[run, t] = cum_regret
            all_arms[run, t] = arm

    return all_regret, all_arms


print("run_experiment listo.")

In [ ]:
T = 1000
N_RUNS = 200

# Define algorithms
algo_specs = [
    ("e-Greedy (e=0.1)", EpsilonGreedy, {"epsilon": 0.1}),
    ("UCB1",             UCB1,           {}),
    ("KL-UCB",           KLUCB,          {}),
    ("Thompson",         ThompsonBernoulli, {}),
    ("EXP3",             EXP3,           {"gamma": np.sqrt(3 * np.log(3) / T)}),
]

algo_colors = [COLORS["gray"], COLORS["blue"], COLORS["green"],
               COLORS["red"], COLORS["purple"]]

# Run on Canonical A (Bernoulli)
print("Ejecutando Canonical A (Bernoulli)...")
results_A = {}
for name, cls, kwargs in algo_specs:
    regret, arms = run_experiment(CANONICAL_A, cls, kwargs, T=T, n_runs=N_RUNS)
    results_A[name] = {"regret": regret, "arms": arms}
    print(f"  {name}: regret medio final = {regret[:, -1].mean():.1f} +/- {regret[:, -1].std():.1f}")

print("\nListo.")

In [ ]:
# Run on Canonical B (Gaussian) — need Gaussian-compatible algorithms
# KL-UCB is Bernoulli-specific; use UCB1 for Gaussian
# Thompson uses ThompsonGaussian for Gaussian rewards

algo_specs_gauss = [
    ("e-Greedy (e=0.1)", EpsilonGreedy, {"epsilon": 0.1}),
    ("UCB1",             UCB1,           {}),
    ("Thompson",         ThompsonGaussian,
     {"mu_prior": 0.0, "sigma_prior": 10.0, "sigma_known": 1.5}),
]

algo_colors_gauss = [COLORS["gray"], COLORS["blue"], COLORS["red"]]

print("Ejecutando Canonical B (Gaussiano)...")
results_B = {}
for name, cls, kwargs in algo_specs_gauss:
    regret, arms = run_experiment(CANONICAL_B, cls, kwargs, T=T, n_runs=N_RUNS)
    results_B[name] = {"regret": regret, "arms": arms}
    print(f"  {name}: regret medio final = {regret[:, -1].mean():.1f} +/- {regret[:, -1].std():.1f}")

print("\nListo.")

In [ ]:
def plot_regret(results, algo_names, colors, title):
    """Plot mean cumulative regret +/- 1 std."""
    fig, ax = plt.subplots(figsize=(10, 6))
    timesteps = np.arange(1, T + 1)

    for name, color in zip(algo_names, colors):
        regret = results[name]["regret"]
        mean = regret.mean(axis=0)
        std = regret.std(axis=0)
        ax.plot(timesteps, mean, color=color, lw=2, label=name)
        ax.fill_between(timesteps, mean - std, mean + std,
                        color=color, alpha=0.15)

    ax.set_xlabel("Ronda (t)")
    ax.set_ylabel("Regret acumulado")
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.legend(fontsize=10)
    ax.set_xlim(1, T)
    fig.tight_layout()
    plt.show()


# Bernoulli
plot_regret(results_A, [n for n, _, _ in algo_specs], algo_colors,
            "Regret acumulado — Canonical A (Bernoulli)")

# Gaussian
plot_regret(results_B, [n for n, _, _ in algo_specs_gauss], algo_colors_gauss,
            "Regret acumulado — Canonical B (Gaussiano)")

---
## Seccion 11: Analisis de seleccion de brazos

Veamos como distribuye cada algoritmo estocastico sus pulls entre los brazos
en tres momentos: $T = 50$, $T = 200$, $T = 1000$.

In [ ]:
check_times = [50, 200, 1000]
stochastic_algos = ["e-Greedy (e=0.1)", "UCB1", "Thompson"]
stochastic_colors = [COLORS["gray"], COLORS["blue"], COLORS["red"]]

fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)

for ax, t_check in zip(axes, check_times):
    x_pos = np.arange(3)  # arms
    width = 0.25

    for j, (name, color) in enumerate(zip(stochastic_algos, stochastic_colors)):
        arms_data = results_A[name]["arms"][:, :t_check]  # (n_runs, t_check)
        # Average fraction of pulls per arm
        fracs = np.array([(arms_data == i).mean() for i in range(3)])
        ax.bar(x_pos + j * width, fracs, width, color=color, label=name, alpha=0.8)

    ax.set_xticks(x_pos + width)
    ax.set_xticklabels([f"Brazo {n}\n($\mu$={m})" for n, m in
                        zip(ARM_NAMES, CANONICAL_A.means)])
    ax.set_title(f"T = {t_check}", fontsize=12)
    ax.set_ylabel("Fraccion de pulls" if ax == axes[0] else "")

axes[0].legend(fontsize=9)
fig.suptitle("Distribucion de pulls por brazo — Canonical A",
             fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

---
## Seccion 12: Porcentaje de pulls optimos en el tiempo

Otra perspectiva: que fraccion de pulls van al brazo optimo (C) conforme avanza
el tiempo. Esta metrica mide la tasa de acierto instantanea.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

optimal_arm = 2  # C is arm index 2
window = 20  # smoothing window

for (name, _, _), color in zip(algo_specs, algo_colors):
    arms_data = results_A[name]["arms"]  # (n_runs, T)
    # Fraction of runs that chose optimal arm at each timestep
    optimal_frac = (arms_data == optimal_arm).mean(axis=0)
    # Smooth with rolling mean
    smoothed = np.convolve(optimal_frac, np.ones(window)/window, mode="valid")
    ax.plot(np.arange(window, T + 1), smoothed, color=color, lw=2, label=name)

ax.set_xlabel("Ronda (t)")
ax.set_ylabel("% pulls al brazo optimo (C)")
ax.set_title("Porcentaje de pulls optimos en el tiempo — Canonical A",
             fontsize=13, fontweight="bold")
ax.axhline(1.0, color="black", ls=":", alpha=0.3, label="100%")
ax.legend(fontsize=10)
ax.set_xlim(1, T)
ax.set_ylim(0, 1.05)
fig.tight_layout()
plt.show()

---
## Seccion 13: Ejercicio de decision

Para cada escenario, elige el algoritmo que usarias y justifica. Despues,
ejecuta la simulacion para verificar.

**Escenario 1: Ensayo clinico** — 3 tratamientos, 50 pacientes total.
Recompensa = recuperacion (Bernoulli). Importa minimizar pacientes con
tratamiento suboptimo. Horizonte muy corto.

**Escenario 2: Recomendacion de noticias** — 5 categorias, usuario cambia
preferencias segun la hora del dia y tendencias. 10000 recomendaciones/dia.
Las recompensas no son estacionarias.

**Escenario 3: A/B test de pagina web** — 2 disenos, 5000 visitantes.
Recompensa = conversion (Bernoulli). Necesitas decidir rapido cual es mejor
y tienes un prior de que ambos estan entre 2% y 10% de conversion.

In [ ]:
mi_eleccion_1 = "Thompson"   # <-- CHANGE THIS (options: EpsilonGreedy, UCB1, KLUCB, Thompson, EXP3)
mi_eleccion_2 = "EXP3"       # <-- CHANGE THIS
mi_eleccion_3 = "Thompson"   # <-- CHANGE THIS

print(f"Escenario 1 (ensayo clinico): {mi_eleccion_1}")
print(f"Escenario 2 (noticias no estacionarias): {mi_eleccion_2}")
print(f"Escenario 3 (A/B test con prior): {mi_eleccion_3}")

In [ ]:
# Scenario 1: Clinical trial — short horizon, Bernoulli
print("=" * 60)
print("Escenario 1: Ensayo clinico (T=50, Bernoulli)")
print("=" * 60)

env_clinical = BanditEnvironment([0.3, 0.5, 0.7], reward_type="bernoulli")
T_clin = 50

for name, cls, kwargs in algo_specs:
    regret, _ = run_experiment(env_clinical, cls, kwargs, T=T_clin, n_runs=N_RUNS)
    print(f"  {name:20s}: regret medio = {regret[:, -1].mean():.2f} "
          f"+/- {regret[:, -1].std():.2f}")

print()
print("=" * 60)
print("Escenario 3: A/B test con prior informativo (T=5000, Bernoulli)")
print("=" * 60)

env_ab = BanditEnvironment([0.04, 0.07], reward_type="bernoulli")

# Thompson with informative prior (alpha=2, beta=30 means prior ~0.06)
class ThompsonInformativePrior:
    def __init__(self, K, alpha0=2, beta0=30):
        self.K = K
        self.alpha = np.full(K, float(alpha0))
        self.beta = np.full(K, float(beta0))
    def select(self, rng):
        samples = rng.beta(self.alpha, self.beta)
        return int(np.argmax(samples))
    def update(self, arm, reward):
        if reward == 1:
            self.alpha[arm] += 1
        else:
            self.beta[arm] += 1

specs_ab = [
    ("e-Greedy (e=0.1)", EpsilonGreedy, {"epsilon": 0.1}),
    ("UCB1",             UCB1,           {}),
    ("Thompson (flat)",  ThompsonBernoulli, {}),
    ("Thompson (prior)", ThompsonInformativePrior, {"alpha0": 2, "beta0": 30}),
]

for name, cls, kwargs in specs_ab:
    regret, _ = run_experiment(env_ab, cls, kwargs, T=5000, n_runs=N_RUNS)
    print(f"  {name:20s}: regret medio = {regret[:, -1].mean():.2f} "
          f"+/- {regret[:, -1].std():.2f}")

### Reflexion sobre los escenarios

- **Escenario 1**: Con horizonte corto ($T=50$), Thompson Sampling converge
  mas rapido que los demas, minimizando pacientes con tratamiento suboptimo.

- **Escenario 2**: Con recompensas no estacionarias, EXP3 es la eleccion
  correcta. Los algoritmos estocasticos asumen distribuciones fijas y sufririan
  regret lineal si las preferencias cambian.

- **Escenario 3**: Thompson con prior informativo aprovecha el conocimiento
  previo (conversiones entre 2-10%) para converger mas rapido que Thompson
  con prior plano.

---
## Reflexion final

En este notebook recorrimos el espectro completo de algoritmos de bandidos:

| Algoritmo | Mecanismo de exploracion | Regret | Cuando usar |
|-----------|--------------------------|--------|-------------|
| $\varepsilon$-Greedy | Aleatorio con prob $\varepsilon$ | $O(\varepsilon T)$ — lineal | Prototipo rapido |
| UCB1 | Bonus de confianza | $O(K \log T / \Delta)$ | Sin hiperparametros |
| KL-UCB | Bonus KL | Lai-Robbins optimo | Bernoulli, maximo rendimiento |
| Thompson | Muestreo del posterior | Lai-Robbins optimo | Caso general, prior disponible |
| EXP3 | Pesos exponenciales | $O(\sqrt{KT \ln K})$ | Entorno adversarial |

**Lecciones clave**:

1. **No hay algoritmo universalmente mejor** — la eleccion depende del contexto
   (estocastico vs adversarial, horizonte, informacion previa).

2. **La exploracion dirigida supera a la aleatoria** — UCB1 y Thompson dominan
   a $\varepsilon$-greedy porque exploran donde hay incertidumbre, no al azar.

3. **El enfoque bayesiano (Thompson) es elegante y practico** — la exploracion
   emerge naturalmente de la incertidumbre del posterior, sin parametros.

4. **Las garantias adversariales tienen un costo** — EXP3 paga $O(\sqrt{T})$
   en vez de $O(\log T)$ por no asumir estocasticidad.

5. **La comparacion empirica complementa la teoria** — los teoremas asintoticos
   no siempre predicen el rendimiento en horizontes finitos.